In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

# import arviz as az
from jax.random import PRNGKey

# import numpyro
# from numpyro import distributions as dist
# from numpyro import infer

pd.options.plotting.backend = "plotly"

from summer3.graph import *
from summer3.epi import *

In [ ]:
disease_state = Stratification("disease_state", ["S", "I", "R"])
humans = CompartmentMap.new(disease_state)
list(c.strata[0][1] for c in humans.compartments)

In [ ]:
from itertools import product

In [ ]:
ll_pairs = list(product(range(16), range(16)))

In [ ]:
age_strat = humans.stratify(Stratification("age", ["child", "adult"]))

In [ ]:
def get_loc_str(ll_pair):
    return f"{ll_pair[0]:02d},{ll_pair[1]:02d}"
loc_strs = [get_loc_str(ll_pair) for ll_pair in ll_pairs]
location = Stratification("location", loc_strs)

In [ ]:
humans.stratify(location)

In [ ]:
location.strata

In [ ]:
age_cats = age_strat.categories()
infectees = location.categories()#age_cats.product(location)
infectors = location.categories()#age_cats.product(location)

In [ ]:
import bidict

In [ ]:
lstratidx = bidict.bidict({k:v for k,v in zip(location.strata, np.arange(len(location.strata)))})

In [ ]:
lstratidx.inv[5]

In [ ]:
mm_raw = np.zeros((len(location.strata),len(location.strata)))

In [ ]:
for lati in range(16):
    for loni in range(16):
        k = f"{lati:02d},{loni:02d}"
        mm_raw[lstratidx[k],lstratidx[k]] = 1.0
        neighbour_keys = [f"{lati+1:02d},{loni:02d}", f"{lati-1:02d},{loni:02d}", f"{lati:02d},{loni+1:02d}", f"{lati:02d},{loni-1:02d}"]
        for nk in neighbour_keys:
            if nk in lstratidx:
                mm_raw[lstratidx[k],lstratidx[nk]] = 0.05
        

In [ ]:
px.imshow(mm_raw)

In [ ]:
mm = mixing_matrix(mm_raw, location.categories(), location.categories())

In [ ]:
# Pass in mm as the last argument if we want heterogeneous mixing; otherwise, InfectionProcess will assume homogeneous mixing.
iprocess = defer(InfectionProcess)(infectees, infectors, disease_state["I"], mm)

In [ ]:
# defer is equivalent to using Function in the old summer2 style, eg the below code is equivalent to
# foi = Function(InfectionProcess.process, [iprocess, CompartmentValues, Parameter("contact_rate", 0.2)])

# Note also that Parameters now take default values

foi = defer(InfectionProcess.process)(
    iprocess, CompartmentValues, Parameter("contact_rate", 0.2)
)

In [ ]:
infection = TransitionFlow("infection", disease_state["S"], disease_state["I"], foi)
recovery = TransitionFlow(
    "recovery",
    disease_state["I"],
    disease_state["R"],
    1.0 / Parameter("recovery_time", 10.0),
)

waning = TransitionFlow("waning", disease_state["R"], disease_state["S"], 1.0 / Parameter("waning_time", 10.0))

In [ ]:
times = pd.date_range("7 jun 1980", "7 december 1980")
epi_model = CompartmentalEpiModel(humans, times)

epi_model.add_flow(infection)
epi_model.add_flow(recovery)
epi_model.add_flow(waning)

In [ ]:
#birth = EntryFlow("birth", age_strat["child"], Parameter("birth_rate", 0.01))
death = ExitFlow("death", disease_state["I"], Parameter("death_rate", 0.1))

#ageing = TransitionFlow("ageing", age_strat["child"], age_strat["adult"], 0.1)

#epi_model.add_flow(birth)
epi_model.add_flow(death)
#epi_model.add_flow(ageing)

In [ ]:
loc_init = np.random.normal(loc=1.0, scale=0.1, size=(16,16))
loc_init[3,3] = 10.0
loc_init[9,11] = 15.0
loc_init = loc_init / loc_init.sum()
loc_init_split = location.categories().wrap(loc_init.flatten())

In [ ]:
pop_data = pd.Series(index=["child", "adult"], data=np.array([1000.0, 1500.0]))
base_pops = strat_data_from_pandas(pop_data, age_strat)
pop_splits = [CategoryData(disease_state.categories(), jnp.array(([1.0, 0.0, 0.0])))]
pop_splits.append(loc_init_split)
epi_model.set_initial_population(base_pops, pop_splits)

In [ ]:
def get_runner(epi_model, params: dict[str, float]):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

In [ ]:
@jit
def seed_pulse(t):
    return jnp.where(0.0 <= t, jnp.where(t < 10.0, 1.0, 0.0), 0.0)

In [ ]:
N_LATS, N_LONS = 16, 16
N_TIMES = len(times)

In [ ]:
def plot_infection_spatial(idata):
    fig = px.imshow(idata, range_color=(0,10.0), animation_frame=0)

    fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 50
    fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 5

    fig.show()

In [ ]:
seed_param = defer(seed_pulse)(Time) * Parameter("seed_rate", 1.0)
seed = EntryFlow("seed", (disease_state["I"], location["04,11"]), seed_param)

epi_model.add_flow(seed)

In [ ]:
params = {
    "contact_rate": 0.4,
    "recovery_time": 10.0,
    "birth_rate": 5.0,
    "death_rate": 0.05,
    "seed_rate": 1.0,
    "waning_time": 30.0,
}
runner, istate = get_runner(epi_model, params)
results = epi_model.run(params)

In [ ]:
spatial_data = results["compartments"].sumcats(compartment=location.categories()).data.reshape(N_TIMES,N_LATS, N_LONS)
idata = results["compartments"].query(compartment=disease_state["I"]).sumcats(compartment=location.categories()).data.reshape(N_TIMES,N_LATS, N_LONS)

In [ ]:
# Infection prevalence over time for whole population

pd.Series((idata/spatial_data).mean(axis=(1,2))).plot()

In [ ]:
plot_infection_spatial(idata)